# 01 · Análise exploratória / Exploratory data analysis

**Objetivo:** entender a estrutura do dataset antes de modelar: tempo de casa (`stag`),
evento de saída (`event`), proporção de **censura** e distribuição das covariáveis.

> Em análise de sobrevivência, funcionários com `event = 0` **não** são "quem ficou para sempre":
> são observações **censuradas à direita**. Sabemos apenas que ficaram *pelo menos* `stag` meses.
> Por isso médias simples de tempo de casa são enviesadas, e é aí que entram os métodos de sobrevivência.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # works even without `pip install -e .`
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from turnover_survival import config, data, nonparametric, models, evaluation

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
T, E = config.DURATION_COL, config.EVENT_COL

df = data.load_clean()
print(f"{len(df)} employees | {df[E].sum()} left | {1 - df[E].mean():.1%} censored")

In [ ]:
df.head()

In [ ]:
df.info()

## Alvo de sobrevivência / Survival target

In [ ]:
display(df[E].value_counts().rename(index={0: "censored (still employed)", 1: "left"}).to_frame("n"))
display(df.groupby(E)[T].describe())

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=df, x=T, hue=E, bins=40, multiple="stack", ax=ax)
ax.set(xlabel="Tenure (months)", title="Tenure distribution by event status")
fig.savefig(config.FIGURES_DIR / "01_tenure_hist.png", dpi=120, bbox_inches="tight")

**Por que não usar a média do `stag` de quem saiu?** Ela ignora os censurados e subestima a
permanência; a mediana de Kaplan-Meier (notebook 02) é o estimador correto.

## Covariáveis categóricas / Categorical covariates

In [ ]:
for col in config.CATEGORICAL_COLS:
    tab = df.groupby(col).agg(n=(E, "size"), turnover_rate=(E, "mean"), median_tenure=(T, "median"))
    display(tab.sort_values("n", ascending=False).style.format({"turnover_rate": "{:.1%}", "median_tenure": "{:.1f}"}).set_caption(col))

> A taxa bruta de saída (`turnover_rate`) **não** controla o tempo de exposição: um grupo com
> funcionários mais novos parece ter menos saídas apenas porque foi observado por menos tempo.
> Use-a só como orientação inicial.

## Covariáveis numéricas / Numeric covariates

In [ ]:
display(df[config.NUMERIC_COLS].describe().T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), config.NUMERIC_COLS):
    sns.boxplot(data=df, x=E, y=col, ax=ax)
    ax.set_title(col)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "01_numeric_by_event.png", dpi=120, bbox_inches="tight")

In [ ]:
corr = df[config.NUMERIC_COLS + [T]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation")
fig.savefig(config.FIGURES_DIR / "01_corr.png", dpi=120, bbox_inches="tight")

## Categorias raras / Rare categories

In [ ]:
rare = {c: df[c].value_counts()[lambda s: s < config.MIN_CATEGORY_FREQ].to_dict() for c in config.CATEGORICAL_COLS}
{k: v for k, v in rare.items() if v}

Categorias com menos de `config.MIN_CATEGORY_FREQ` observações serão agrupadas em `other` na
matriz de modelagem (`data.make_design_matrix`) para evitar coeficientes instáveis no Cox.

### Conclusões / Takeaways
> _Preencha após rodar com os dados reais. / Fill in after running on the real data._